In [ ]:
# AlphaMissense conected code

#poznamka pro dominika - poradie metod je AlphaMissense, Poly-Phen-2 a SIFT na záver

In [ ]:
# Running PyMissense from: https://github.com/MPI-Dortmund/pymissense
# We assign a UniProt ID to each protein according to this: 
# protein name UniProtID
# GLUT1	P11166
# GLUT2	P11168
# GLUT3	P11169
# GLUT4	P14672
# GLUT5	P22732
# GLUT6	Q9UGQ3
# GLUT7	Q6PXP3
# GLUT8	Q9NY64
# GLUT9	Q9NRM0
# GLUT10 O95528
# GLUT11 Q9BYW1
# GLUT12 Q8TD20
# GLUT13 Q96QE2
# GLUT14 Q8TDB8 

In [ ]:
pip install pymissense

In [ ]:
# For each protein, it is necessary to run the command separately 
!pymissense --tsv "C:/Users/Nina/Desktop/AlphaMissense - ProjektSLC/AlphaMissense_aa_substitutions.tsv/AlphaMissense_aa_substitutions.tsv" \
    --pdbpath "C:/Users/Nina/Desktop/Counted AlphaMissense/Q96QE2.pdb" Q96QE2 \
    "C:/Users/Nina/Desktop/Counted AlphaMissense"

In [ ]:
# From counted results of pathogenicity will be counted average of pathogenicity for the whole protein
sum = 0
count = 0
file = "..GLUT project documentation/Results from AlphaMissense/UniProtID-edit.pdb" 
for line in open(f'{file}'):
        list = line.split()
        id = list[0]
        if id == 'ATOM':
            sum += float(list[10])
            count += 1

print(sum/count)
# Result will be writen down to the single AlpaMissense patogenicity.txt file. For each protein will be a number of average pathogenicity.
# Results can be find here /GLUT project documentation/Results from AlphaMissense/AlpaMissense patogenicity.txt/

In [ ]:
# Detection of protein parts - extracellular domain, intracellular domain and transmembrane region. 
# Done though: https://dtu.biolib.com/DeepTMHMM/ for each protein.
# Results have been separated and stored in .xlsx file for each protein separately.
# Results can be find here GLUT project documentation/Results from DeepTMHMM/PyMissense"

In [ ]:
# Protein regions - processing of results
import pandas as pd
# protein name_sq files are part of files deeptmhmm_results, which can be find for each protein in a zip file
# This cell have to be run for each protein separately
# 1. Loading of the original file
file_path = "../GLUT project documentation/Results from DeepTMHMM/Sequences/protein name_sq.txt"  # <- protein sequence file
with open(file_path, "r") as f:
    lines = f.read().splitlines()

sequence = lines[0].strip()
annotations = lines[1].strip()

# 2. Control of the protein lenght
if len(sequence) != len(annotations):
    raise ValueError("Leght of sequence and the annotation are not the same!")

# 3. Creating a list I (in the cell), M (membrane region), O (out from the cell)
I_list, M_list, O_list = [], [], []

for i, (aa, tag) in enumerate(zip(sequence, annotations), start=1):
    entry = {"position": i, "name": aa}
    if tag == 'I':
        I_list.append(entry)
    elif tag == 'M':
        M_list.append(entry)
    elif tag == 'O':
        O_list.append(entry)

# 4. DataFrames creation and alignment by number of lines
max_len = max(len(I_list), len(M_list), len(O_list))

def pad_list(lst):
    return lst + [{"position": "", "name": ""}] * (max_len - len(lst))

I_df = pd.DataFrame(pad_list(I_list))
M_df = pd.DataFrame(pad_list(M_list))
O_df = pd.DataFrame(pad_list(O_list))

# 5. Renaming columns. Position - position of the amino acid in the sequence, Name - type of the amino acid in alphabetical abbreviations
I_df.columns = ["I_position", "I_name"]
M_df.columns = ["M_position", "M_name"]
O_df.columns = ["O_position", "O_name"]

# 6. Combining into one table
final_df = pd.concat([O_df, M_df, I_df], axis=1)

# 7. Save to Excel
output_file = "protein name_AMK_output_MIO.xlsx"
final_df.to_excel(output_file, index=False)

print(f"The resulting file was saved as: {output_file}")
# Saved in /GLUT project documentation/Results from DeepTMHMM/Excel output from sequences
# After processing Excel files, prepare in files column for Pathogenicity

In [ ]:
#Protein regions - pathogenicity assignment. The cell have to be run for each protein separately. 
import pandas as pd

# Loading an Excel file
df = pd.read_excel("../GLUT project documentation/Results from DeepTMHMM/Excel output from sequences/protein name_AMK_output_MIO.xlsx")

df.columns = [col.strip() for col in df.columns]

# Converting positions to integers (if they are floats or text)
for col in ['O_position', 'M_position', 'I_position']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64') #this was added because of Excel string problem

# Loading a PDB file counted by PyMissense
with open("../GLUT project documentation/Results from AlphaMissense/UniProtID-edit.pdb", "r") as f:
    pdb_lines = f.readlines()

# Parsing residues and pathogenicity from column 11
residue_patho = {}
for line in pdb_lines:
    if line.startswith("ATOM"):
        try:
            resnum = int(line[22:26].strip())
            patho = float(line[60:66].strip())
            if resnum not in residue_patho:
                residue_patho[resnum] = patho
        except ValueError:
            continue  # ignoruj zle parsované riadky

# Function to obtain pathogenicity for a given position
def get_patho(pos):
    return residue_patho.get(int(pos)) if pd.notnull(pos) else None

# Add pathogenicity to all three positions
df['O_patogenicity'] = df['O_position'].apply(get_patho)
df['M_patogenicity'] = df['M_position'].apply(get_patho)
df['I_patogenicity'] = df['I_position'].apply(get_patho)

# Output
df.to_excel("output_with_pathogenicity_PyMissenseMIO_protein name.xlsx", index=False)
df.head()
# Outputs saved in /GLUT project documentation/Results from DeepTMHMM/PyMissense/

In [ ]:
#The pathogenicity averages for individual protein parts were calculated in a newly created .xlsx file: /output_with_pathogenicity_PyMissenseMIO_"protein name".xlsx/ using the AVERAGE function. 
#The calculated average was copied to a .txt file using the following code:
import pandas as pd
import os

# Set the path to the folder with the files
folder_path = "../GLUT project documentation/Results from DeepTMHMM/PyMissense"  # A folder containing all newly created files with calculated averages

# Initialization of the list for results
results = []

# Going through all files in the folder
for file in os.listdir(folder_path):
    if file.endswith(".xlsx"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_excel(file_path)
        
        # Find a row with the value 'AVERAGE' in any column
        avg_row = df[df.apply(lambda row: row.astype(str).str.contains("AVERAGE", case=False).any(), axis=1)]
        
        if not avg_row.empty:
            row = avg_row.iloc[0]
            values = [
                row.get("O_patogenicity", None),
                row.get("M_patogenicity", None),
                row.get("I_patogenicity", None)
            ]
            results.append([file, *values])

# Conversion to DataFrame
result_df = pd.DataFrame(results, columns=["filename", "O_patogenicity", "M_patogenicity", "I_patogenicity"])

# Save to text file
result_df.to_csv("PyMissense(MIO).txt", sep="\t", index=False)
# Saved here /GLUT project documentation/Results from DeepTMHMM/PyMissense(MIO).txt/
# Showing the output
print(result_df.head())

In [ ]:
# The amino acid residues lining the pores of individual proteins were calculated using: https://mole.upol.cz/ -LINING RESIDUES
# The amino acid residues framing the protein binding site were calculated using: https://prankweb.cz/ - BINDING PLACE
# The results were transcribed into an .xlsx document. Saved here /GLUT project documentation/Binding places and lining residues/Excel files/
# After transcription, a third column was created, where only those amino acid residues that were not part of the binding site but framed the protein pore were transcribed. -BINDING PLACE-LINING RESIDUES

In [ ]:
#To select the correct binding site from the calculated set, we will use the following code:
import pandas as pd
import os

# Path to the folder with results from PrankWeb
folder_path = r"../GLUT project documentation/Results from PrankWeb/Binding_places_counted"

for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)

        try:
            # Reading a CSV file with a TAB separator
            df = pd.read_csv(file_path, sep=';')

            # Checking if 'pocket' exists
            if 'pocket' not in df.columns:
                print(f"✖ Súbor {filename} neobsahuje stĺpec 'pocket'.")
                continue

            # Filtering rows where pocket == 1
            filtered = df[df['pocket'] == 1]

            if not filtered.empty:
                lines = filtered.apply(
                    lambda row: f"{int(row['residue'])}\t{row['residue_label'].strip()} 1", axis=1
                )

                txt_name = os.path.splitext(filename)[0] + ".txt"
                txt_path = os.path.join(folder_path, txt_name)

                with open(txt_path, 'w', encoding='utf-8') as f:
                    f.write('\n'.join(lines))

                print(f"✔ Saved: {txt_path}")
            else:
                print(f"⚠️ File {filename} do not contain pocket == 1.")

        except Exception as e:
            print(f"✖ Misstake {filename}: {e}")

print("✅ Done.")

In [ ]:
#Further processing of results in .xlsx files containing the lining residues, binding places and lining residues without biding places
import pandas as pd

# Loading an Excel file
df = pd.read_excel("../GLUT project documentation/Binding places and lining residues/Excel files/protein name analysis lr_bp_lr-bp.xlsx")

# Renaming columns
df.columns = [col.strip() for col in df.columns]

# Converting positions to integers (if they are floats or text)
for col in ['lr_position', 'bp_position', 'lr-bp_position']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# Loading a PDB file counted by PyMissense
with open("../GLUT project documentation/Results from AlphaMissense/UniProtID-edit.pdb", "r") as f:
    pdb_lines = f.readlines()

# Parsing residues and pathogenicity from column 11
residue_patho = {}
for line in pdb_lines:
    if line.startswith("ATOM"):
        try:
            resnum = int(line[22:26].strip())
            patho = float(line[60:66].strip())
            if resnum not in residue_patho:
                residue_patho[resnum] = patho
        except ValueError:
            continue  # ignoruj zle parsované riadky

# Function to obtain pathogenicity for a given position
def get_patho(pos):
    return residue_patho.get(int(pos)) if pd.notnull(pos) else None

# Add pathogenicity to all three positions
df['lr_patogenicity'] = df['lr_position'].apply(get_patho)
df['bp_patogenicity'] = df['bp_position'].apply(get_patho)
df['lr-bp_patogenicity'] = df['lr-bp_position'].apply(get_patho)

# Output
df.to_excel("output_with_pathogenicity_PyMissense_protein name.xlsx", index=False)
df.head()

In [ ]:
#The pathogenicity averages for individual protein parts were calculated in a newly created .xlsx file: /output_with_pathogenicity_PyMissense_"protein name".xlsx/ using the AVERAGE function. 
#The calculated average was copied to a .txt file using the following code:
import pandas as pd
import os

# Set the path to the folder with the files
folder_path = "GLUT project documentation/Binding places and lining residues/PyMissense"  # A folder containing all newly created files with calculated averages

# Initialization of the list for results
results = []

# Going through all files in the folder
for file in os.listdir(folder_path):
    if file.endswith(".xlsx"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_excel(file_path)
        
        # Find a row with the value 'AVERAGE' in any column
        avg_row = df[df.apply(lambda row: row.astype(str).str.contains("AVERAGE", case=False).any(), axis=1)]
        
        if not avg_row.empty:
            row = avg_row.iloc[0]
            values = [
                row.get("lr_patogenicity", None),
                row.get("bp_patogenicity", None),
                row.get("lr-bp_patogenicity", None)
            ]
            results.append([file, *values])

# Conversion to DataFrame
result_df = pd.DataFrame(results, columns=["filename", "lr_patogenicity", "bp_patogenicity", "lr-bp_patogenicity"])

# Save to text file
result_df.to_csv("PyMissense(lr,bp,lr-bp).txt", sep="\t", index=False)
# Saved here /GLUT project documentation/Binding places and lining residues/PyMissense/PyMissenseGLUT(lr,bp,lr-bp).txt/ 
# Showing the output
print(result_df.head())

In [ ]:
#Finally, we merged all three newly created .txt files together

In [ ]:
import pandas as pd
# Loading three files
df1 = pd.read_csv("../GLUT project documentation/Results from AlphaMissense/AlpaMissense patogenicity.txt", sep="\t")  
df2 = pd.read_csv("../GLUT project documentation/Results from DeepTMHMM/PyMissense/PyMissense(MIO).txt", sep="\t")
df3 = pd.read_csv("../GLUT project documentation/Binding places and lining residues/PyMissense/PyMissenseGLUT(lr,bp,lr-bp).txt", sep="\t")

# Merge data according to the common column 'filename'. This column contains the names of proteins.
merged_df = df1.merge(df2, on="protein", how="outer")
merged_df = merged_df.merge(df3, on="protein", how="outer")

# Saving the resulting file
merged_df.to_csv("connected_file_PyMissense.txt", sep="\t", index=False)


In [ ]:
#Creating the final heatmap
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Loading file
df = pd.read_csv("../GLUT project documentation/Conected results/connected_file_PyMissense.txt", sep="\t")

# Setting 'filename' as index
df.set_index("protein", inplace=True)

# Creating a heat map with values
plt.figure(figsize=(10, len(df) * 0.4))
sns.heatmap(
    df,
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    linecolor='gray',
    annot=True,
    fmt=".3f"  
)


plt.title("GLUTs pathogenicity profile Alpha Missense", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()


plt.savefig("heatmap_patogenicityPyMissense.png", dpi=300, bbox_inches='tight')

plt.show()